<a href="https://colab.research.google.com/github/mishra-yogendra/intent_expansion_pipeline/blob/main/intent_expansion_pipeline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers umap-learn scikit-learn faiss-cpu requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 59.2 MB/s eta 0:00:00


In [ ]:
import os, json, re, time, math
from typing import List, Dict, Any, Tuple
from collections import Counter, defaultdict

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
import faiss
import requests

In [ ]:

CONFIG = {
    # Embeddings
    "embedding_model": "all-mpnet-base-v2",
    "embedding_batch_size": 64,

    # Sampling / scale
    "max_points_for_full_clustering": 20000,  # cluster on all points below this
    "sample_size_for_clustering": 50000,
    # Dimensionality reduction
    "use_umap": True,
    "umap_components": 10,
    "umap_n_neighbors": 15,
    "svd_components": 64,

    # MiniBatchKMeans k-search
    "k_min": 3,
    "k_max": 20,

    # Files
    "data_path": "/content/Data.json",
    "out_dir": "out",
}

In [ ]:

import getpass

if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API key: ")

print("Groq API key loaded into environment (not printed).")

Enter Groq API key: ··········
Groq API key loaded into environment (not printed).


In [ ]:
def load_data(path: str):
    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)
    return j.get("intent_mapper", []), j.get("customer_messages", [])

def flatten_rec(rec: Dict) -> str:
    parts = []
    if rec.get("history"):
        parts.append(rec["history"])
    if rec.get("current_human_message"):
        parts.append(rec["current_human_message"])
    return "\n".join([p for p in parts if p]).strip()

def normalize_text(t: str) -> str:
    t = (t or "").lower()
    t = re.sub(r'https?://\S+', ' ', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

In [ ]:
_SBM_CACHE = {}

def get_model(name: str) -> SentenceTransformer:
    if name not in _SBM_CACHE:
        _SBM_CACHE[name] = SentenceTransformer(name)
    return _SBM_CACHE[name]

def embed_texts(texts: List[str], model_name: str, batch_size: int = 64) -> np.ndarray:
    model = get_model(model_name)
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return emb

def choose_sample_indices(n: int) -> np.ndarray:
    if n <= CONFIG["max_points_for_full_clustering"]:
        return np.arange(n)
    rng = np.random.default_rng(42)
    m = min(CONFIG["sample_size_for_clustering"], n)
    return rng.choice(n, size=m, replace=False)

In [ ]:
def reduce_dimensionality(X: np.ndarray) -> np.ndarray:
    if CONFIG["use_umap"]:
        try:
            import umap
            n_comp = min(CONFIG["umap_components"], X.shape[1])
            um = umap.UMAP(
                n_components=n_comp,
                n_neighbors=CONFIG["umap_n_neighbors"],
                metric="cosine",
                random_state=42,
            )
            return um.fit_transform(X)
        except Exception:
            pass
    # Fallback: TruncatedSVD
    n_comp = min(CONFIG["svd_components"], max(2, X.shape[1] // 2))
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    red = svd.fit_transform(X)
    return normalize(red)

In [ ]:
def cluster_sample_minibatch(X_red_sample: np.ndarray) -> Tuple[np.ndarray, int, float]:
    """
    MiniBatchKMeans with small k-search, chosen by silhouette on the sample. [web:64][web:109]
    """
    n = X_red_sample.shape[0]
    k_min = CONFIG["k_min"]
    k_max = min(CONFIG["k_max"], max(k_min + 1, n // 20))

    best = {"k": None, "score": -1.0, "labels": None}
    for k in range(k_min, k_max + 1):
        mbk = MiniBatchKMeans(
            n_clusters=k,
            batch_size=2048,
            init_size=min(10000, n),
            n_init="auto",
            random_state=42,
        )
        labels_k = mbk.fit_predict(X_red_sample)
        if len(set(labels_k)) < 2:
            continue
        try:
            s = silhouette_score(X_red_sample, labels_k)
        except Exception:
            s = -1.0
        if s > best["score"]:
            best = {"k": k, "score": s, "labels": labels_k}
    if best["k"] is None:
        return np.zeros(n, dtype=int), 1, -1.0
    print(f"[INFO] MiniBatchKMeans best k={best['k']}, silhouette={best['score']:.3f}")
    return best["labels"].astype(int), best["k"], best["score"]

In [ ]:
def compute_centroids(emb_full: np.ndarray, sample_idx: np.ndarray, labels_sample: np.ndarray, k: int) -> np.ndarray:
    centroids = np.zeros((k, emb_full.shape[1]), dtype=np.float32)
    for c in range(k):
        mask = labels_sample == c
        if not np.any(mask):
            continue
        points = emb_full[sample_idx[mask]]
        centroids[c] = points.mean(axis=0)
    norms = np.linalg.norm(centroids, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    centroids = centroids / norms
    return centroids

In [ ]:
def assign_clusters_faiss(emb_full: np.ndarray, centroids: np.ndarray, batch_size: int = 8192) -> np.ndarray:
    d = emb_full.shape[1]
    index = faiss.IndexFlatIP(d)  # cosine via inner product on normalized vectors
    index.add(centroids.astype(np.float32))

    n = emb_full.shape[0]
    labels = np.empty(n, dtype=np.int32)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = emb_full[start:end].astype(np.float32)
        _, idx = index.search(batch, 1)
        labels[start:end] = idx[:, 0]
    return labels

In [ ]:
def sample_metrics(X_red_sample: np.ndarray, labels_sample: np.ndarray) -> Dict[str, Any]:
    info = {"global_silhouette": None, "per_cluster": {}}
    try:
        if len(set(labels_sample)) >= 2:
            g = float(silhouette_score(X_red_sample, labels_sample))
            info["global_silhouette"] = g
            s_samples = silhouette_samples(X_red_sample, labels_sample)
            for c in sorted(set(labels_sample)):
                mask = labels_sample == c
                info["per_cluster"][int(c)] = float(s_samples[mask].mean())
    except Exception as e:
        info["error"] = str(e)
    return info

def compute_coherence(emb_full: np.ndarray, labels_full: np.ndarray, max_points_per_cluster: int = 300) -> Dict[int, float]:
    out = {}
    rng = np.random.default_rng(42)
    for c in sorted(set(labels_full)):
        idx = np.where(labels_full == c)[0]
        if len(idx) <= 1:
            out[int(c)] = 0.0
            continue
        if len(idx) > max_points_per_cluster:
            idx = rng.choice(idx, size=max_points_per_cluster, replace=False)
        sub = emb_full[idx]
        sim = cosine_similarity(sub)
        n = sim.shape[0]
        sum_upper = (sim.sum() - np.trace(sim)) / 2.0
        pairs = n * (n - 1) / 2.0
        out[int(c)] = float(sum_upper / pairs) if pairs > 0 else 0.0
    return out

In [ ]:
def groq_chat_completion(messages: List[Dict[str, str]], model: str = "llama-3.1-8b-instant") -> Tuple[str, str]:
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return None, "GROQ_API_KEY not set in environment"
    candidates = [
        os.environ.get("GROQ_BASE_URL", "").rstrip("/")
        if os.environ.get("GROQ_BASE_URL") else None,
        "https://api.groq.com/openai/v1",
    ]
    last_err = None
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": 220,
        "temperature": 0.0,
    }
    for base in candidates:
        if not base:
            continue
        url = base.rstrip("/") + "/chat/completions"
        try:
            r = requests.post(
                url,
                headers={
                    "Authorization": f"Bearer {api_key}",
                    "Content-Type": "application/json",
                },
                json=payload,
                timeout=30,
            )
            r.raise_for_status()
            j = r.json()
            if "choices" in j and len(j["choices"]) > 0:
                content = (
                    j["choices"][0].get("message", {}).get("content")
                    or j["choices"][0].get("text")
                    or json.dumps(j["choices"][0])
                )
                return content, None
            else:
                last_err = f"Unexpected response: {str(j)[:400]}"
        except requests.HTTPError as e:
            last_err = f"HTTPError {e.response.status_code}: {e.response.text[:400]}"
            continue
        except Exception as e:
            last_err = f"{type(e).__name__}: {str(e)}"
            continue
    return None, last_err or "No Groq base endpoints succeeded"

In [ ]:
def parse_llm_json(text: str) -> Any:
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'(\{[\s\S]*\})', text)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                pass
    res = {}
    for line in text.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            res[k.strip()] = v.strip()
    return res or {"raw": text}

In [ ]:
def build_groq_messages(top_terms: List[str], sample_messages: List[str]) -> List[Dict[str, str]]:
    examples_text = "\n".join(
        f"{i+1}) {m}" for i, m in enumerate(sample_messages[:5])
    )
    return [
        {
            "role": "system",
            "content": (
                "You are an intent taxonomy assistant. "
                "Return ONLY a single JSON object with keys:\n"
                '"label" (2-5 words, clear intent),\n'
                '"id" (snake_case),\n'
                '"description" (<=25 words),\n'
                '"confidence" (0-1 number).\n'
                "No extra commentary, no markdown, no code fences."
            ),
        },
        {
            "role": "user",
            "content": (
                "Top terms for this cluster: " + ", ".join(top_terms[:12]) +
                "\n\nExample user messages:\n" + examples_text
            ),
        },
    ]

In [ ]:
def top_tfidf_terms(docs_norm: List[str], labels: np.ndarray, top_n: int = 12) -> Dict[int, List[str]]:
    tf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
    X = tf.fit_transform(docs_norm)
    terms = tf.get_feature_names_out()
    clusters_terms = {}
    for lab in sorted(set(labels)):
        idx = np.where(labels == lab)[0]
        if len(idx) == 0:
            clusters_terms[int(lab)] = []
            continue
        mean_vec = X[idx].mean(axis=0).A1
        top = mean_vec.argsort()[-top_n:][::-1]
        clusters_terms[int(lab)] = [terms[i] for i in top]
    return clusters_terms

In [ ]:
def guess_primary_from_terms(terms: List[str]) -> str:
    t = " ".join(terms).lower()
    if any(w in t for w in ["order", "track", "delivery", "refund", "cancel"]):
        return "logistics"
    if any(w in t for w in ["hair", "serum", "shampoo", "oil", "ingredients", "price"]):
        return "specific_product"
    if any(w in t for w in ["recommend", "suggest", "best product"]):
        return "recommendation"
    if any(w in t for w in ["hello", "hi", "thanks", "thank you"]):
        return "basic_interactions"
    return "unknown"

In [ ]:
def run_intent_discovery_pipeline(
    data_path: str = None,
    out_dir: str = None,
    use_groq_chat: bool = True,
) -> Dict[str, Any]:
    data_path = data_path or CONFIG["data_path"]
    out_dir = out_dir or CONFIG["out_dir"]
    os.makedirs(out_dir, exist_ok=True)

    # 1) Load data
    intent_mapper, messages = load_data(data_path)

    raw, norm = [], []
    for rec in messages:
        t = flatten_rec(rec)
        if not t:
            continue
        raw.append(t)
        norm.append(normalize_text(t))
    n = len(norm)
    print(f"[INFO] Loaded {n} messages.")

    # 2) Embeddings (batched)
    emb_full = embed_texts(norm, CONFIG["embedding_model"], CONFIG["embedding_batch_size"])

    # 3) Sample for clustering
    sample_idx = choose_sample_indices(n)
    print(f"[INFO] Using {len(sample_idx)} messages for clustering sample.")
    emb_sample = emb_full[sample_idx]

    # 4) Dimensionality reduction on sample
    X_red_sample = reduce_dimensionality(emb_sample)

    # 5) Cluster sample with MiniBatchKMeans
    labels_sample, k, sil_sample = cluster_sample_minibatch(X_red_sample)

    # 6) Centroids + assign full dataset with FAISS
    centroids = compute_centroids(emb_full, sample_idx, labels_sample, k)
    labels_full = assign_clusters_faiss(emb_full, centroids)

    # 7) Metrics
    sil_info = sample_metrics(X_red_sample, labels_sample)
    coherence = compute_coherence(emb_full, labels_full)

    cluster_sizes = {int(c): int(cnt) for c, cnt in Counter(labels_full).items()}
    print("[INFO] Cluster sizes:", cluster_sizes)
    print("[INFO] Global silhouette (sample):", sil_info.get("global_silhouette"))

    # 8) Existing secondary names for novelty
    existing_secondary = []
    for p in intent_mapper:
        for s in p.get("secondary_intents", []):
            existing_secondary.append(s.get("name", ""))
    existing_set = set([s.lower() for s in existing_secondary])

    # 9) TF-IDF terms per cluster
    clusters_terms = top_tfidf_terms(norm, labels_full, top_n=12)

    # 10) Build proposals
    proposals = []
    min_cluster_size_absolute = 6
    novel_terms_threshold = 2
    min_size_for_proposal = 10
    min_confidence_for_proposal = 0.3

    for lab, terms in clusters_terms.items():
        size = cluster_sizes.get(int(lab), 0)
        if size < min_cluster_size_absolute:
            continue

        novel_terms = [
            t for t in terms
            if not any(t in ex for ex in existing_set)
        ]
        if len(novel_terms) < novel_terms_threshold:
            continue

        full_idxs = [i for i, l in enumerate(labels_full) if l == lab]
        samples = [raw[i] for i in full_idxs[:6]]

        cluster_sil = sil_info.get("per_cluster", {}).get(int(lab))
        coh = coherence.get(int(lab), 0.0)
        conf = 0.7 * (cluster_sil if cluster_sil is not None else 0.0) + 0.3 * coh

        if size < min_size_for_proposal or conf < min_confidence_for_proposal:
            continue

        proposal = {
            "cluster": int(lab),
            "size": int(size),
            "cluster_share": float(size) / float(n) if n > 0 else 0.0,
            "top_terms": terms,
            "novel_terms": novel_terms,
            "sample_messages": samples,
            "cluster_silhouette": cluster_sil,
            "coherence": float(coh),
            "proposal_confidence": float(conf),
            "suggested_primary": guess_primary_from_terms(terms),
        }

        if use_groq_chat:
            messages_prompt = build_groq_messages(terms, samples)
            text, err = groq_chat_completion(messages_prompt)
            if err:
                proposal["llm_error"] = err
            else:
                parsed = parse_llm_json(text) or {}
                if isinstance(parsed, dict):
                    if "id" in parsed:
                        parsed["id"] = re.sub(r"[^a-z0-9_]", "_", parsed["id"].lower())
                    if "confidence" in parsed:
                        try:
                            c = float(parsed["confidence"])
                            parsed["confidence"] = max(0.0, min(1.0, c))
                        except Exception:
                            pass
                proposal["llm_raw"] = text
                proposal["llm_parsed"] = parsed

        proposals.append(proposal)

    out = {
        "num_messages": n,
        "cluster_sizes": cluster_sizes,
        "sample_silhouette": sil_info,
        "coherence": coherence,
        "proposals": proposals,
    }

    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "proposals.json"), "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)
    print("[DONE] Outputs ->", out_dir)
    print(f"[INFO] Proposals generated: {len(proposals)}")
    return out

In [ ]:
out = run_intent_discovery_pipeline()
len(out["proposals"]), out["cluster_sizes"], out["sample_silhouette"].get("global_silhouette")

[INFO] Loaded 200 messages.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[INFO] Using 200 messages for clustering sample.


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[INFO] MiniBatchKMeans best k=9, silhouette=0.586
[INFO] Cluster sizes: {8: 17, 1: 16, 6: 23, 7: 20, 2: 21, 4: 38, 3: 37, 5: 22, 0: 6}
[INFO] Global silhouette (sample): 0.5859825015068054
[DONE] Outputs -> out
[INFO] Proposals generated: 8


(8,
 {8: 17, 1: 16, 6: 23, 7: 20, 2: 21, 4: 38, 3: 37, 5: 22, 0: 6},
 0.5859825015068054)